# SQL Analysis — Logistics Optimisation

**Author:** Kresio Azevedo Fernando  
**Portfolio:** [kresio-azevedo-fernando.github.io](https://kresio-azevedo-fernando.github.io)  

---

## Business Problem

Warehouse with **3,204 products** across 5 categories.  
Service level at **80%** — generating **14,746 stockouts/month**.  
Estimated revenue loss: **€630 million/year**.

This notebook answers three questions using SQL:
1. **What happened?** — Where are the stockouts concentrated?
2. **Why did it happen?** — What is driving the service level failure?
3. **What to do?** — Which products and zones should be prioritised?

---

## Setup — Install dependencies and create sample database

In [ ]:
# Install required libraries
!pip install pandas matplotlib seaborn -q

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import numpy as np

# Chart styling
plt.rcParams.update({
    'figure.facecolor': '#09090f',
    'axes.facecolor':   '#0f0f1a',
    'axes.labelcolor':  '#e8e8f0',
    'xtick.color':      '#9494a8',
    'ytick.color':      '#9494a8',
    'text.color':       '#e8e8f0',
    'axes.titlecolor':  '#e8e8f0',
    'axes.edgecolor':   '#1a1a28',
    'grid.color':       '#1a1a28',
    'axes.grid':        True,
})
ACCENT = '#bb9476'
print('✓ Setup complete')

In [ ]:
# Create sample database that mirrors the real dataset structure
# In production: replace with ETL pipeline output (logistics.db)

np.random.seed(42)
conn = sqlite3.connect(':memory:')

CATEGORIES = ['Pharma', 'Electronics', 'Food', 'Industrial', 'Other']
ZONES      = ['A', 'B', 'C', 'D', 'E']

# Products table
n_products = 3204
products = pd.DataFrame({
    'product_id':   [f'P{str(i).zfill(4)}' for i in range(1, n_products+1)],
    'category':     np.random.choice(CATEGORIES, n_products,
                        p=[0.25, 0.24, 0.20, 0.18, 0.13]),
    'zone':         np.random.choice(ZONES, n_products),
    'turnover':     np.random.choice(['A','B','C'], n_products,
                        p=[0.20, 0.30, 0.50]),
    'storage_cost': np.round(np.random.uniform(1.5, 5.0, n_products), 2),
    'lead_time_days': np.random.randint(2, 12, n_products),
})

# Stockouts table (last month)
n_stockouts = 14746
stockout_probs = {'Pharma': 0.28, 'Electronics': 0.25,
                  'Food': 0.18, 'Industrial': 0.14, 'Other': 0.15}
cats_weighted = np.random.choice(
    CATEGORIES, n_stockouts,
    p=[stockout_probs[c] for c in CATEGORIES]
)
stockouts = pd.DataFrame({
    'stockout_id':     range(1, n_stockouts+1),
    'product_id':      np.random.choice(products['product_id'], n_stockouts),
    'category':        cats_weighted,
    'zone':            np.random.choice(ZONES, n_stockouts),
    'revenue_lost':    np.round(np.random.uniform(200, 1800, n_stockouts), 2),
    'demand_units':    np.random.randint(1, 50, n_stockouts),
    'week':            np.random.randint(1, 5, n_stockouts),
})

# Stock movements table
n_movements = 25000
movements = pd.DataFrame({
    'movement_id':   range(1, n_movements+1),
    'product_id':    np.random.choice(products['product_id'], n_movements),
    'category':      np.random.choice(CATEGORIES, n_movements),
    'movement_type': np.random.choice(['IN','OUT'], n_movements, p=[0.45,0.55]),
    'units':         np.random.randint(1, 200, n_movements),
    'cost_per_unit': np.round(np.random.uniform(1.5, 5.0, n_movements), 2),
    'week':          np.random.randint(1, 53, n_movements),
})

products.to_sql('products',   conn, index=False, if_exists='replace')
stockouts.to_sql('stockouts', conn, index=False, if_exists='replace')
movements.to_sql('movements', conn, index=False, if_exists='replace')

print(f'✓ Database ready')
print(f'  products:  {len(products):,} rows')
print(f'  stockouts: {len(stockouts):,} rows')
print(f'  movements: {len(movements):,} rows')

---
## Query 1 — DIAGNOSTIC: Where are stockouts concentrated?

**Business question:** Which categories and zones are driving the 14,746 monthly stockouts?

In [ ]:
query_1 = """
SELECT
    category,
    COUNT(*)                          AS total_stockouts,
    ROUND(SUM(revenue_lost), 0)       AS total_revenue_lost,
    ROUND(AVG(revenue_lost), 0)       AS avg_loss_per_stockout,
    ROUND(
        COUNT(*) * 100.0 /
        (SELECT COUNT(*) FROM stockouts), 1
    )                                 AS pct_of_total
FROM stockouts
GROUP BY category
ORDER BY total_stockouts DESC;
"""

df1 = pd.read_sql(query_1, conn)
print('STOCKOUTS BY CATEGORY')
print(df1.to_string(index=False))

In [ ]:
# Visualise Query 1
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Stockouts by category
bars = axes[0].barh(df1['category'], df1['total_stockouts'], color=ACCENT)
axes[0].set_title('Stockouts by Category', fontsize=12, pad=10)
axes[0].set_xlabel('Number of Stockouts')
for bar, val in zip(bars, df1['total_stockouts']):
    axes[0].text(bar.get_width() + 50, bar.get_y() + bar.get_height()/2,
                 f'{val:,}', va='center', fontsize=9, color='#9494a8')

# Revenue loss by category
bars2 = axes[1].barh(df1['category'], df1['total_revenue_lost']/1e6,
                      color='#6eb5ff')
axes[1].set_title('Revenue Lost by Category (€M)', fontsize=12, pad=10)
axes[1].set_xlabel('Revenue Lost (€M)')
for bar, val in zip(bars2, df1['total_revenue_lost']/1e6):
    axes[1].text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
                 f'€{val:.1f}M', va='center', fontsize=9, color='#9494a8')

plt.tight_layout()
plt.suptitle('Query 1 — Stockout Concentration by Category',
             y=1.02, fontsize=13, color='white')
plt.savefig('q1_stockouts_by_category.png', dpi=150,
            bbox_inches='tight', facecolor='#09090f')
plt.show()

top2 = df1.head(2)
print(f"\nINSIGHT: {top2['category'].iloc[0]} and {top2['category'].iloc[1]}"
      f" account for {top2['pct_of_total'].sum():.0f}% of all stockouts.")
print("   These two categories must be prioritised in stock reallocation.")

---
## Query 2 — DIAGNOSTIC: What is driving the failure?

**Business question:** Is the problem demand variability, layout inefficiency, or lead time?

In [ ]:
query_2 = """
SELECT
    p.category,
    p.zone,
    p.turnover,
    ROUND(AVG(p.lead_time_days), 1)   AS avg_lead_time,
    ROUND(AVG(p.storage_cost), 2)     AS avg_storage_cost,
    COUNT(s.stockout_id)              AS stockouts,
    ROUND(SUM(s.revenue_lost), 0)     AS revenue_lost
FROM products p
LEFT JOIN stockouts s
    ON p.product_id = s.product_id
GROUP BY p.category, p.zone, p.turnover
ORDER BY stockouts DESC
LIMIT 15;
"""

df2 = pd.read_sql(query_2, conn)
print('STOCKOUTS BY CATEGORY + ZONE + TURNOVER')
print(df2.to_string(index=False))

# Key finding: high-turnover products in distant zones
high_turn = df2[df2['turnover'] == 'A'].copy()
print(f"\nINSIGHT: High-turnover (A) products average "
      f"{high_turn['avg_lead_time'].mean():.1f} days lead time.")
print("   These products should be relocated to zones closest to dispatch.")

---
## Query 3 — DIAGNOSTIC: Demand variability analysis

**Business question:** How volatile is demand per category? (Standard deviation vs mean)

In [ ]:
query_3 = """
SELECT
    category,
    ROUND(AVG(demand_units), 1)                          AS avg_demand,
    ROUND(
        (
            SELECT AVG((d.demand_units - sub.mean) * (d.demand_units - sub.mean))
            FROM stockouts d
            JOIN (
                SELECT category, AVG(demand_units) AS mean
                FROM stockouts
                GROUP BY category
            ) sub ON d.category = sub.category
            WHERE d.category = s.category
        ), 2
    )                                                    AS demand_variance,
    COUNT(*)                                             AS observations
FROM stockouts s
GROUP BY category
ORDER BY demand_variance DESC;
"""

# Simpler equivalent using pandas (SQLite lacks STDDEV)
query_3_simple = """
SELECT
    category,
    ROUND(AVG(demand_units), 1)  AS avg_demand,
    ROUND(AVG(revenue_lost), 0)  AS avg_revenue_lost,
    COUNT(*)                     AS observations
FROM stockouts
GROUP BY category
ORDER BY avg_revenue_lost DESC;
"""

df3_base = pd.read_sql(query_3_simple, conn)

# Calculate std deviation in pandas
std_df = stockouts.groupby('category')['demand_units'].std().round(1)
df3 = df3_base.merge(std_df.rename('demand_std'), on='category')
df3['cv_pct'] = (df3['demand_std'] / df3['avg_demand'] * 100).round(1)

print('DEMAND VARIABILITY BY CATEGORY')
print(df3.to_string(index=False))

max_cv = df3.loc[df3['cv_pct'].idxmax()]
print(f"\n🔍 INSIGHT: {max_cv['category']} has the highest demand variability "
      f"(CV = {max_cv['cv_pct']}%).")
print("   High variability = safety stock must be increased for this category.")

---
## Query 4 — OPTIMISATION: Which products to reprioritise?

**Business question:** Which specific products generate the most losses and should be reallocated first?

In [ ]:
query_4 = """
SELECT
    s.category,
    s.zone,
    COUNT(s.stockout_id)             AS stockout_count,
    ROUND(SUM(s.revenue_lost), 0)    AS total_loss,
    ROUND(AVG(p.storage_cost), 2)    AS avg_storage_cost,
    ROUND(AVG(p.lead_time_days), 1)  AS avg_lead_time,
    CASE
        WHEN COUNT(s.stockout_id) > 800
             AND AVG(p.lead_time_days) > 7
        THEN 'CRITICAL — Relocate + Reduce Lead Time'
        WHEN COUNT(s.stockout_id) > 600
        THEN 'HIGH — Increase Safety Stock'
        WHEN COUNT(s.stockout_id) > 400
        THEN 'MEDIUM — Monitor Weekly'
        ELSE 'LOW — Standard Process'
    END AS action
FROM stockouts s
JOIN products p ON s.product_id = p.product_id
GROUP BY s.category, s.zone
ORDER BY total_loss DESC
LIMIT 12;
"""

df4 = pd.read_sql(query_4, conn)
print('PRIORITISATION MATRIX — Category × Zone')
print(df4.to_string(index=False))

critical = df4[df4['action'].str.startswith('CRITICAL')]
print(f"\n🔍 INSIGHT: {len(critical)} category-zone combinations classified as CRITICAL.")
print(f"   These represent the highest-priority reallocation targets.")

In [ ]:
# Heatmap: Revenue loss by Category × Zone
pivot = df4.pivot_table(
    values='total_loss',
    index='category',
    columns='zone',
    aggfunc='sum',
    fill_value=0
)

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    pivot / 1000,
    annot=True, fmt='.0f',
    cmap='YlOrBr',
    linewidths=0.5,
    linecolor='#1a1a28',
    ax=ax,
    cbar_kws={'label': 'Revenue Lost (€K)'}
)
ax.set_title('Revenue Loss Heatmap — Category × Zone (€K)',
             fontsize=12, pad=12)
ax.set_xlabel('Zone')
ax.set_ylabel('Category')
plt.tight_layout()
plt.savefig('q4_heatmap.png', dpi=150,
            bbox_inches='tight', facecolor='#09090f')
plt.show()
print('📈 Heatmap saved: q4_heatmap.png')

---
## Summary — Business Conclusions

From the SQL analysis:

In [ ]:
total_loss = stockouts['revenue_lost'].sum()
top_cat    = df1.iloc[0]
top2_pct   = df1.head(2)['pct_of_total'].sum()

print('=' * 60)
print(' BUSINESS CONCLUSIONS — SQL ANALYSIS')
print('=' * 60)
print(f"""
1. WHAT HAPPENED
   14,746 stockouts last month across 3,204 products.
   Total estimated revenue lost: €{total_loss/1e6:.1f}M (monthly).
   Annualised: ~€{total_loss*12/1e6:.0f}M in revenue at risk.

2. WHERE IT HAPPENED
   {top_cat['category']} leads with {top_cat['total_stockouts']:,} stockouts
   ({top_cat['pct_of_total']}% of total).
   Top 2 categories = {top2_pct:.0f}% of all stockouts.

3. WHY IT HAPPENED
   • Demand variability: CV > 50% in high-priority categories.
   • Layout inefficiency: high-turnover products in distant zones.
   • Lead time: avg 6+ days for critical categories.

4. WHAT TO DO (Prescriptive — via Simplex + Dijkstra)
   • Reallocate 23% of surplus stock to top-2 categories.
   • Relocate Zone A products to zones closest to dispatch.
   • Reduce routes from 177 to 51 via Dijkstra optimisation.
   • Target: lead time 6.0 → 4.3 days, service level 80% → 94%.

   Projected financial impact: +€320M vs. baseline (ROI 420%)
""")
print('=' * 60)
print('Next step: run simplex_model.py and dijkstra_routes.py')
print('Portfolio: kresio-azevedo-fernando.github.io')